In [1]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

# warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.append(root_path)

In [2]:
from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=True)

2025-04-08 17:15:26,409 - root - ERROR - Error writing configs: [Errno 2] No such file or directory: '/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/conf/conf_client.yml'
Traceback (most recent call last):
  File "/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/hummingbot/client/config/config_helpers.py", line 876, in save_to_yml
    with open(yml_path, "w", encoding="utf-8") as outfile:
FileNotFoundError: [Errno 2] No such file or directory: '/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/conf/conf_client.yml'


In [ ]:
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
import datetime
from decimal import Decimal
from controllers.directional_trading.pz_scalper import PZScalperControllerConfig


# Controller configuration
connector_name = "binance_perpetual"
trading_pair = "WLD-USDT"
interval = "5m"
backtesting_resolution = "1m"

# Don't matter
cooldown_time = 1 #60 * 15
take_profit = 5 # 100%, -> Disable Take profit, let the trailing do it's job
stop_loss = 5
trailing_stop_activation_price = 1
trailing_stop_trailing_delta = 0.05

# General
total_amount_quote = 1000
max_executors_per_side = 2


# Indicator Values
hma_fast: int = 25
hma_slow: int = 30
natr_length = 13

# Triple Barrier
time_limit = 1200
tp_natr_factor = 1.5
sl_natr_factor = 3
ts_activation_natr_factor = 0.5
ts_delta_natr_factor = 0.25
###


# Creating the instance of the configuration and the controller
config = PZScalperControllerConfig(
    connector_name=connector_name,
    trading_pair=trading_pair,
    interval=interval,
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    trailing_stop=TrailingStop(activation_price=Decimal(trailing_stop_activation_price), trailing_delta=Decimal(trailing_stop_trailing_delta)),
    total_amount_quote=Decimal(total_amount_quote),
    time_limit=time_limit,
    max_executors_per_side=max_executors_per_side,
    cooldown_time=cooldown_time,
    natr_length = natr_length,
    sl_natr_factor=sl_natr_factor,
    ts_activation_natr_factor = ts_activation_natr_factor,
    ts_delta_natr_factor = ts_delta_natr_factor,
    tp_natr_factor=tp_natr_factor,
    hma_fast=hma_fast,
    hma_slow=hma_slow,
)

In [ ]:
# Running the backtesting this will output a backtesting result object that has built in methods to visualize the results

start = int(datetime.datetime(2025, 3, 20).timestamp())
end = int(datetime.datetime(2025, 3, 30).timestamp())

backtesting_result = await backtesting.run_backtesting(config, start, end, backtesting_resolution)

2025-04-07 17:46:09,401 - hummingbot.connector.time_synchronizer - WARNING - Could not refresh server time. Check network connection.
2025-04-07 17:46:09,402 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7372b4084e80>
2025-04-07 17:46:09,404 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x7373346289a0>, 76644.63490428)])']
connector: <aiohttp.connector.TCPConnector object at 0x7372b4084e20>


OSError: Error executing request GET https://fapi.binance.com/fapi/v1/exchangeInfo. HTTP status is 418. Error: {"code":-1003,"msg":"Way too many requests; IP(62.216.201.50) banned until 1744041230132. Please use the websocket for live updates to avoid bans."}

In [ ]:

# candles_df = backtesting_result.processed_data
# candles_df

In [ ]:
import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# Let's see what is inside the backtesting results
print(backtesting_result.get_results_summary())
fig = backtesting_result.get_backtesting_figure()
# Add EMAs
candles_df = backtesting_result.processed_data

fast_key = f"HMA_{hma_fast}"
slow_key = f"HMA_{hma_slow}"


fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[fast_key],
                         line=dict(color='#00FF00', width=2),
                         name='Fast HMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[slow_key],
                         line=dict(color='#FF0000', width=2),
                         name='Fast HMA'))



Net PNL: $-130.84 (-13.08%) | Max Drawdown: $-150.79 (-15.15%)
Total Volume ($): 544000.00 | Sharpe Ratio: -1.73 | Profit Factor: 0.80
Total Executors: 544 | Accuracy Long: 0.40 | Accuracy Short: 0.42
Close Types: Take Profit: 164 | Stop Loss: 260 | Time Limit: 120 |
             Trailing Stop: 0 | Early Stop: 0



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Grab processed data
candles_df = backtesting_result.processed_data

# Create subplots layout
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    row_heights=[0.60, 0.15, 0.15, 0.1],
    vertical_spacing=0.02,
    subplot_titles=["Price + Indicators", "Condition", "Crossover"]
)

# Row 1: Close price
fig.add_trace(go.Scatter(
    x=candles_df.index, y=candles_df["close"],
    line=dict(color='#FFFFFF', width=2), name='Close'), row=1, col=1)


hma_fast_key = f"HMA_{hma_fast}"
hma_slow_key = f"HMA_{hma_slow}"
# ema_medium_key = f"EMA_{ema_medium}"
# ema_slow_key = f"EMA_{ema_slow}"


fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[hma_fast_key],
                         line=dict(color='#00FF00', width=2),
                         name='Fast HMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[hma_slow_key],
                         line=dict(color='#FF0000', width=2),
                         name='Slow HMA'))
# fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_medium_key],
#                          line=dict(color='#FFA500', width=2),
#                          name='Slow HMA'))
# fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_slow_key],
#                          line=dict(color='#FFFFFF', width=2),
#                          name='Slow EMA'))

# Row 4: Signal values as line + markers
# fig.add_trace(go.Scatter(
#     x=candles_df.index,
#     y=candles_df[f"STOCHRSId_{srsi_length}_{srsi_length}_{srsi_smoothing}_{srsi_smoothing}"],
#     mode='lines',
#     line=dict(color='red', width=2),
#     name='short_condition',
#     marker=dict(size=6)
# ), row=2, col=1)
# fig.add_trace(go.Scatter(
#     x=candles_df.index,
#     y=candles_df[f"STOCHRSIk_{srsi_length}_{srsi_length}_{srsi_smoothing}_{srsi_smoothing}"],
#     mode='lines',
#     line=dict(color='green', width=2),
#     name='short_condition',
#     marker=dict(size=6)
# ), row=2, col=1)

# fig.add_trace(go.Scatter(
#     x=candles_df.index,
#     y=candles_df["bearish_crossover"],
#     mode='lines+markers',
#     line=dict(color='cyan', width=2),
#     name='bearish_crossover',
#     marker=dict(size=6)
# ), row=3, col=1)
fig.add_trace(go.Scatter(
    x=candles_df.index,
    y=candles_df["signal"],
    mode='lines+markers',
    line=dict(color='cyan', width=2),
    name='signal',
    marker=dict(size=6)
), row=3, col=1)


# Final layout
fig.update_layout(
    height=700,
    title="Backtest with Candles, HMAs, RSI, and StochRSI",
    showlegend=True,
    template="plotly_dark"
)

fig.show()



In [ ]:
# # 2. The executors dataframe: this is the dataframe that contains the information of the orders that were executed
import pandas as pd

executors_df = backtesting_result.executors_df
executors_df

,id,timestamp,type,close_timestamp,close_type,status,config,net_pnl_pct,net_pnl_quote,cum_fees_quote,filled_amount_quote,is_active,is_trading,custom_info,controller_id,side
0,AAvB7n2S3yCkDz89XEsfjuacPcW5rjNnf2xxyWvhMka7,1740799200,position_executor,1740799800,CloseType.STOP_LOSS,RunnableStatus.TERMINATED,{'id': 'AAvB7n2S3yCkDz89XEsfjuacPcW5rjNnf2xxyW...,-0.0088033714955378476740843751713327947072684...,-4.4016857477689237398976729309652000665664672...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 1.1184, 'level_id': None, 'sid...",None,SELL
1,CBaUfmzDQX3KcVSRjtwKS55uifnJWfdGDYHnY8Rj2bHe,1740804900,position_executor,1740805200,CloseType.STOP_LOSS,RunnableStatus.TERMINATED,{'id': 'CBaUfmzDQX3KcVSRjtwKS55uifnJWfdGDYHnY8...,-0.0057827361272451797968852105213954928331077...,-2.8913680636225898012980906059965491294860839...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 1.1133, 'level_id': None, 'sid...",None,BUY
2,6CrYbdAMA51yeFY52CVNWCFKWyMAXHrPrBG1zyEbh26W,1740805500,position_executor,1740806400,CloseType.TAKE_PROFIT,RunnableStatus.TERMINATED,{'id': '6CrYbdAMA51yeFY52CVNWCFKWyMAXHrPrBG1zy...,0.00425786254048203874345013275615201564505696...,2.12893127024101946886958103277720510959625244...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 1.1062, 'level_id': None, 'sid...",None,SELL
3,8qSZdvFdY29t8QdfnibjF1H72eqi2ziTEfgUg2FJQPYc,1740813900,position_executor,1740814200,CloseType.TAKE_PROFIT,RunnableStatus.TERMINATED,{'id': '8qSZdvFdY29t8QdfnibjF1H72eqi2ziTEfgUg2...,0.00450297065791887066166232500563637586310505...,2.25148532895943542797567715751938521862030029...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 1.103, 'level_id': None, 'side...",None,BUY
4,9HEuAc9EEyvp9gzYEsY3YMgiv64d9eQAMUECF4DzAWub,1740819000,position_executor,1740820200,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': '9HEuAc9EEyvp9gzYEsY3YMgiv64d9eQAMUECF4...,-0.0008690582959639086809214281181823480437742...,-0.4345291479819543556395444738882360979914665...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 1.1153, 'level_id': None, 'sid...",None,SELL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
539,8mSE1G9weqm2Vr4mMYKn96aCHPAsJ99dYT24Vpibq8Ag,1743345900,position_executor,1743345900,CloseType.STOP_LOSS,RunnableStatus.TERMINATED,{'id': '8mSE1G9weqm2Vr4mMYKn96aCHPAsJ99dYT24Vp...,-0.0005999999999999999474378786779027450393186...,-0.2999999999999999888977697537484345957636833...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 0.786, 'level_id': None, 'side...",None,BUY
540,FbhLXTYxMdu1b7h44LFsHrDv3rT3pNSc7LbTnaAh6roG,1743350100,position_executor,1743351300,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': 'FbhLXTYxMdu1b7h44LFsHrDv3rT3pNSc7LbTna...,-0.0000879672299028516320798520844448376010404...,-0.0439836149514258242798625531122524989768862...,0.29999999999999998889776975374843459576368331...,1000.0000000000001136868377216160297393798828125,False,False,"{'close_price': 0.7808, 'level_id': None, 'sid...",None,SELL
541,HxpTyefksrL2uaYqVTJMmwftP7FoB7c9GF2KsrrC47au,1743355800,position_executor,1743357000,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': 'HxpTyefksrL2uaYqVTJMmwftP7FoB7c9GF2Ksr...,-0.0009850596842510654649327506149347755126655...,-0.4925298421255327463441631152818445116281509...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 0.7788, 'level_id': None, 'sid...",None,BUY
542,5ADwzDuX5yXHdUmTQG3FeKgK6mjhgaieqqGHctfqewQ4,1743362700,position_executor,1743363900,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': '5ADwzDuX5yXHdUmTQG3FeKgK6mjhgaieqqGHct...,0.00143123016376771675095547209366486640647053...,0.71561508188385836159994823901797644793987274...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 0.7861, 'level_id': None, 'sid...",None,

### Backtesting Analysis

### Scatter of PNL per Trade
This bar chart illustrates the PNL for each individual trade. Positive PNLs are shown in green and negative PNLs in red, providing a clear view of profitable vs. unprofitable trades.


In [ ]:
# import plotly.express as px

# # Create a new column for profitability
# executors_df['profitable'] = executors_df['net_pnl_quote'] > 0

# # Create the scatter plot
# fig = px.scatter(
#     executors_df,
#     x="timestamp",
#     y='net_pnl_quote',
#     title='PNL per Trade',
#     color='profitable',
#     color_discrete_map={True: 'green', False: 'red'},
#     labels={'timestamp': 'Timestamp', 'net_pnl_quote': 'Net PNL (Quote)'},
#     hover_data=['filled_amount_quote', 'side']
# )

# # Customize the layout
# fig.update_layout(
#     xaxis_title="Timestamp",
#     yaxis_title="Net PNL (Quote)",
#     legend_title="Profitable",
#     font=dict(size=12, color="white"),
#     showlegend=False,
#     plot_bgcolor='rgba(0,0,0,0.8)',  # Dark background
#     paper_bgcolor='rgba(0,0,0,0.8)',  # Dark background for the entire plot area
#     xaxis=dict(gridcolor="gray"),
#     yaxis=dict(gridcolor="gray")
# )

# # Add a horizontal line at y=0 to clearly separate profits and losses
# fig.add_hline(y=0, line_dash="dash", line_color="lightgray")

# # Show the plot
# fig.show()

### Histogram of PNL Distribution
The histogram displays the distribution of PNL values across all trades. It helps in understanding the frequency and range of profit and loss outcomes.


In [ ]:
# fig = px.histogram(executors_df, x='net_pnl_quote', title='PNL Distribution')
# fig.show()


# Conclusion
We can see that the indicator has potential to bring good signals to trade and might be interesting to see how we can design a market maker that shifts the mid price based on this indicator.
A lot of the short signals are wrong but if we zoom in into the loss signals we can see that the losses are not that big and the wins are bigger and if we had implemented the trailing stop feature probably a lot of them are going to be profits.

# Next steps
- Filter only the loss signals and understand what you can do to prevent them
- Try different configuration values for the indicator
- Test in multiple markets, pick mature markets like BTC-USDT or ETH-USDT and also volatile markets like DOGE-USDT or SHIB-USDT